In [1]:
import joblib

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from gensim.models import Word2Vec

import re
import pymorphy3
from razdel import tokenize
from nltk.corpus import stopwords

In [2]:
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Предобработка данных

In [3]:
df = pd.read_csv(r'data\small_petitions.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   public_petition_text  3000 non-null   object
 1   reason_category       3000 non-null   object
dtypes: object(2)
memory usage: 47.0+ KB


In [4]:
df['reason_category'].value_counts()

reason_category
Благоустройство                                                                     1764
Содержание МКД                                                                       714
Нарушение правил пользования общим имуществом                                         99
Незаконная информационная и (или) рекламная конструкция                               86
Фасад                                                                                 67
Повреждения или неисправность элементов уличной инфраструктуры                        58
Кровля                                                                                47
Состояние рекламных или информационных конструкций                                    41
Водоснабжение                                                                         41
Незаконная реализация товаров с торгового оборудования (прилавок, ящик, с земли)      18
Санитарное состояние                                                                  18
Центр

In [5]:
morph = pymorphy3.MorphAnalyzer()
russian_stopwords = set(stopwords.words('russian'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^а-яё\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def tokenize_with_razdel(text):
    return [token.text for token in tokenize(text)]

def lemmatize_tokens(tokens):
    lemmas = []
    for token in tokens:
        parsed = morph.parse(token)[0]
        lemma = parsed.normal_form
        lemmas.append(lemma)
    return lemmas

def remove_stopwords(tokens):
    return [token for token in tokens if token not in russian_stopwords and len(token) > 2]

def full_text_preprocessing(text):
    cleaned_text = preprocess_text(text)
    tokens = tokenize_with_razdel(cleaned_text)
    lemmas = lemmatize_tokens(tokens)
    filtered_lemmas = remove_stopwords(lemmas)
    
    return filtered_lemmas

In [6]:
df['processed_tokens'] = df['public_petition_text'].apply(full_text_preprocessing)
df.head()

,public_petition_text,reason_category,processed_tokens
0,На газоне разбросан различный мусор. \r\nПрось...,Благоустройство,"[газон, разбросать, различный, мусор, просьба,..."
1,"Конструкции, препятствующие парковке",Благоустройство,"[конструкция, препятствовать, парковка]"
2,Лестницу убирают раз в три месяца и просто выл...,Содержание МКД,"[лестница, убирать, месяц, просто, выливать, в..."
3,мусор с задней стороны дома,Благоустройство,"[мусор, задний, сторона, дом]"
4,Снова не работает уличный фонарь у входной две...,Содержание МКД,"[снова, работать, уличный, фонарь, входной, дв..."


In [7]:
texts = df['processed_tokens'].tolist()
texts[:3]

[['газон',
  'разбросать',
  'различный',
  'мусор',
  'просьба',
  'вывезти',
  'утилизация',
  'координата'],
 ['конструкция', 'препятствовать', 'парковка'],
 ['лестница',
  'убирать',
  'месяц',
  'просто',
  'выливать',
  'ведро',
  'вода',
  'это',
  'уборка']]

In [8]:
w2v_model = Word2Vec(sentences=texts, vector_size=100, window=5, min_count=3, workers=8, sg=1, epochs=50, seed=RANDOM_SEED)
print(f"Обучено векторов: {len(w2v_model.wv.key_to_index)}")

Обучено векторов: 1547


In [9]:
def qualitative_evaluation(model, test_words):
    for word in test_words:
        if word in model.wv:
            similar = model.wv.most_similar(word, topn=3)
            print(f"Слова, связанные с '{word}':")
            for similar_word, score in similar:
                print(f"  {similar_word}: {score:.3f}")
            print()
        else:
            print(f"Слово '{word}' не найдено в словаре\n")

test_words = ['ремонт', 'мусор', 'дом', 'уборка', 'вода', 'крыша']
qualitative_evaluation(w2v_model, test_words)

Слова, связанные с 'ремонт':
  капитальный: 0.621
  косметический: 0.620
  начать: 0.567

Слова, связанные с 'мусор':
  пластик: 0.631
  мелкий: 0.613
  окурок: 0.609

Слова, связанные с 'дом':
  загородный: 0.513
  парашютный: 0.474
  гсанктпетербург: 0.467

Слова, связанные с 'уборка':
  влажный: 0.661
  подметание: 0.659
  подметать: 0.651

Слова, связанные с 'вода':
  холодный: 0.616
  горячий: 0.614
  напор: 0.603

Слова, связанные с 'крыша':
  протекать: 0.627
  несмотря: 0.609
  неоднократный: 0.586



In [10]:
w2v_model.save('w2v_sg_small.bin')

In [11]:
df['token_vectors'] = df['processed_tokens'].apply(lambda tokens: [w2v_model.wv[token] if token in w2v_model.wv else np.zeros(100) for token in tokens])

In [12]:
df.head()

,public_petition_text,reason_category,processed_tokens,token_vectors
0,На газоне разбросан различный мусор. \r\nПрось...,Благоустройство,"[газон, разбросать, различный, мусор, просьба,...","[[-0.30997294, -0.1431102, -0.22329657, -0.082..."
1,"Конструкции, препятствующие парковке",Благоустройство,"[конструкция, препятствовать, парковка]","[[-0.15901472, -0.52917826, -0.29479125, 0.508..."
2,Лестницу убирают раз в три месяца и просто выл...,Содержание МКД,"[лестница, убирать, месяц, просто, выливать, в...","[[0.14848511, -0.1815279, -0.6278617, -0.38801..."
3,мусор с задней стороны дома,Благоустройство,"[мусор, задний, сторона, дом]","[[0.13902979, -0.16689232, 0.4760844, -0.30861..."
4,Снова не работает уличный фонарь у входной две...,Содержание МКД,"[снова, работать, уличный, фонарь, входной, дв...","[[0.6262138, -0.04213814, -0.13157123, 0.03187..."


In [13]:
le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['reason_category'])

joblib.dump(le, 'label_encoder.pkl')
df.head()

,public_petition_text,reason_category,processed_tokens,token_vectors,category_encoded
0,На газоне разбросан различный мусор. \r\nПрось...,Благоустройство,"[газон, разбросать, различный, мусор, просьба,...","[[-0.30997294, -0.1431102, -0.22329657, -0.082...",0
1,"Конструкции, препятствующие парковке",Благоустройство,"[конструкция, препятствовать, парковка]","[[-0.15901472, -0.52917826, -0.29479125, 0.508...",0
2,Лестницу убирают раз в три месяца и просто выл...,Содержание МКД,"[лестница, убирать, месяц, просто, выливать, в...","[[0.14848511, -0.1815279, -0.6278617, -0.38801...",11
3,мусор с задней стороны дома,Благоустройство,"[мусор, задний, сторона, дом]","[[0.13902979, -0.16689232, 0.4760844, -0.30861...",0
4,Снова не работает уличный фонарь у входной две...,Содержание МКД,"[снова, работать, уличный, фонарь, входной, дв...","[[0.6262138, -0.04213814, -0.13157123, 0.03187...",11


In [14]:
for i, category in enumerate(le.classes_):
    print(f"{category} : {i}")

Благоустройство : 0
Водоотведение : 1
Водоснабжение : 2
Кровля : 3
Нарушение порядка пользования общим имуществом : 4
Нарушение правил пользования общим имуществом : 5
Незаконная информационная и (или) рекламная конструкция : 6
Незаконная реализация товаров с торгового оборудования (прилавок, ящик, с земли) : 7
Повреждения или неисправность элементов уличной инфраструктуры : 8
Подвалы : 9
Санитарное состояние : 10
Содержание МКД : 11
Состояние рекламных или информационных конструкций : 12
Фасад : 13
Центральное отопление : 14


In [15]:
df.to_csv('data/all_data.csv')

In [16]:
df = df[['token_vectors', 'category_encoded']].copy()
df.head()

,token_vectors,category_encoded
0,"[[-0.30997294, -0.1431102, -0.22329657, -0.082...",0
1,"[[-0.15901472, -0.52917826, -0.29479125, 0.508...",0
2,"[[0.14848511, -0.1815279, -0.6278617, -0.38801...",11
3,"[[0.13902979, -0.16689232, 0.4760844, -0.30861...",0
4,"[[0.6262138, -0.04213814, -0.13157123, 0.03187...",11


In [17]:
X = df['token_vectors']
y = df['category_encoded']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

len(X_train), len(X_val), len(X_test)

(2100, 450, 450)

# Создание моделей

In [18]:
device = torch.device('cuda')

In [19]:
def prepare_data(X, y, max_len=50, vector_size=100):
    processed_sequences = np.zeros((len(X), max_len, vector_size), dtype=np.float32)
    
    for i, sequence in enumerate(X):
        current_len = min(len(sequence), max_len)

        if len(sequence) > 0:
            sequence_array = np.array(sequence[:current_len], dtype=np.float32)
            processed_sequences[i, :current_len] = sequence_array

    X_tensor = torch.from_numpy(processed_sequences)
    y_tensor = torch.tensor(y, dtype=torch.long)
    
    return X_tensor, y_tensor


X_tensor, y_tensor = prepare_data(X_train, y_train.values)
X_tensor = X_tensor.to(device)
y_tensor = y_tensor.to(device)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

X_tensor_val, y_tensor_val = prepare_data(X_val, y_val.values)
X_tensor_val = X_tensor_val.to(device)
y_tensor_val = y_tensor_val.to(device)

dataset_val = TensorDataset(X_tensor_val, y_tensor_val)
dataloader_val = DataLoader(dataset_val, batch_size=32, shuffle=False)

In [20]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        _, hidden = self.rnn(x)
        return self.fc(hidden.squeeze(0))

class SimpleLSTM(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden.squeeze(0))

class SimpleGRU(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.fc(hidden.squeeze(0))

In [21]:
class CustomGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.W_r = nn.Linear(input_size, hidden_size)
        self.U_r = nn.Linear(hidden_size, hidden_size)

        self.W_z = nn.Linear(input_size, hidden_size)
        self.U_z = nn.Linear(hidden_size, hidden_size)

        self.W_h = nn.Linear(input_size, hidden_size)
        self.U_h = nn.Linear(hidden_size, hidden_size)

    def forward(self, x, h_prev=None):
        batch_size, seq_len, _ = x.size()
        
        if h_prev is None:
            h_prev = torch.zeros(batch_size, self.hidden_size).to(x.device)
        
        outputs = []
        h = h_prev
        
        for t in range(seq_len):
            x_t = x[:, t, :]

            r = torch.sigmoid(self.W_r(x_t) + self.U_r(h))
            z = torch.sigmoid(self.W_z(x_t) + self.U_z(h))
            h_tilde = torch.tanh(self.W_h(x_t) + self.U_h(r * h))
            h = (1 - z) * h + z * h_tilde
            outputs.append(h.unsqueeze(1))
        
        return torch.cat(outputs, dim=1), h

class CustomGRUModel(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.gru = CustomGRU(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        _, hidden = self.gru(x)
        return self.fc(hidden)

In [ ]:
class AttentionLayer(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, all_hidden_states, final_hidden_state):
        h_n = final_hidden_state
        e = torch.bmm(all_hidden_states, h_n.unsqueeze(2)).squeeze(2)
        alpha = F.softmax(e, dim=1)
        context = torch.bmm(alpha.unsqueeze(1), all_hidden_states).squeeze(1)
        c = torch.cat([h_n, context], dim=1)
        return c

In [ ]:
class SimpleRNNWithAttention(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.attention = AttentionLayer()
        self.fc = nn.Linear(2 * hidden_size, num_classes)
    
    def forward(self, x):
        out, hidden = self.rnn(x)
        final_hidden = hidden.squeeze(0)
        c = self.attention(out, final_hidden)
        output = self.fc(c)
        return output

class SimpleLSTMWithAttention(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.attention = AttentionLayer()
        self.fc = nn.Linear(2 * hidden_size, num_classes)
    
    def forward(self, x):
        out, (hidden, _) = self.lstm(x)
        final_hidden = hidden.squeeze(0)
        c = self.attention(out, final_hidden)
        output = self.fc(c)
        return output

class SimpleGRUWithAttention(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.hidden_size = hidden_size
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.attention = AttentionLayer()
        self.fc = nn.Linear(2 * hidden_size, num_classes)
    
    def forward(self, x):
        out, hidden = self.gru(x)
        final_hidden = hidden.squeeze(0)
        c = self.attention(out, final_hidden)
        output = self.fc(c)
        return output

class CustomGRUWithAttention(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.hidden_size = hidden_size
        self.gru = CustomGRU(input_size, hidden_size)
        self.attention = AttentionLayer()
        self.fc = nn.Linear(2 * hidden_size, num_classes)
    
    def forward(self, x):
        out, hidden = self.gru(x)
        c = self.attention(out, hidden)
        output = self.fc(c)
        return output

In [ ]:
def predict_classes_from_X(model, X_data):
    X_tensor, _ = prepare_data(X_data, [])
    X_tensor = X_tensor.to(device)
    
    with torch.no_grad():
        outputs = model(X_tensor)
        _, preds = torch.max(outputs, 1)
    
    return preds.cpu().numpy()

def calculate_f1(model, dataloader):
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            outputs = model(batch_X)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())
    
    return f1_score(all_targets, all_preds, average='weighted')

def train_model(model, train_loader, val_loader, epochs=10, model_name='Model'):
    model.to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters())
    best_f1 = 0
    
    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

        current_f1 = calculate_f1(model, val_loader)

        if current_f1 > best_f1:
            best_f1 = current_f1
            torch.save(model.state_dict(), f'best_{model_name}.pth')
            print(f'!-- New best model --! F1: {current_f1:.4f}')
        
        if epoch % 10 == 0:
            print(f'Epoch {epoch+1}, F1: {current_f1:.4f}')

    model.load_state_dict(torch.load(f'best_{model_name}.pth', weights_only=False))
    
    return model

In [25]:
rnn_model = SimpleRNN()
lstm_model = SimpleLSTM()
gru_model = SimpleGRU()
custom_gru_model = CustomGRUModel().to(device)
rnn_attention_model = SimpleRNNWithAttention().to(device)
lstm_attention_model = SimpleLSTMWithAttention().to(device)
gru_attention_model = SimpleGRUWithAttention().to(device)
custom_gru_attention_model = CustomGRUWithAttention().to(device)

In [26]:
train_model(rnn_model, dataloader, dataloader_val, epochs=70, model_name="RNN")
train_model(lstm_model, dataloader, dataloader_val, epochs=70, model_name="LSTM") 
train_model(gru_model, dataloader, dataloader_val, epochs=70, model_name="GRU")
train_model(custom_gru_model, dataloader, dataloader_val, epochs=70, model_name="CustomGRU")
train_model(rnn_attention_model, dataloader, dataloader_val, epochs=70, model_name="RNN_Attention")
train_model(lstm_attention_model, dataloader, dataloader_val, epochs=70, model_name="LSTM_Attention")
train_model(gru_attention_model, dataloader, dataloader_val, epochs=70, model_name="GRU_Attention")
train_model(custom_gru_attention_model, dataloader, dataloader_val, epochs=70, model_name="CustomGRU_Attention")

!-- New best model --! F1: 0.4205
Epoch 1, F1: 0.4205
!-- New best model --! F1: 0.4310
!-- New best model --! F1: 0.4339
!-- New best model --! F1: 0.4357
!-- New best model --! F1: 0.4405
Epoch 11, F1: 0.4363
Epoch 21, F1: 0.4319
!-- New best model --! F1: 0.4406
!-- New best model --! F1: 0.6117
!-- New best model --! F1: 0.6427
Epoch 31, F1: 0.4352
Epoch 41, F1: 0.4293
!-- New best model --! F1: 0.6493
!-- New best model --! F1: 0.6627
!-- New best model --! F1: 0.6644
!-- New best model --! F1: 0.6739
!-- New best model --! F1: 0.6930
Epoch 51, F1: 0.6930
!-- New best model --! F1: 0.7028
!-- New best model --! F1: 0.7035
!-- New best model --! F1: 0.7049
!-- New best model --! F1: 0.7232
!-- New best model --! F1: 0.7366
Epoch 61, F1: 0.7365
!-- New best model --! F1: 0.7439
!-- New best model --! F1: 0.7546
!-- New best model --! F1: 0.7615
!-- New best model --! F1: 0.4205
Epoch 1, F1: 0.4205
!-- New best model --! F1: 0.4261
!-- New best model --! F1: 0.4315
!-- New best model

CustomGRUWithAttention(
  (gru): CustomGRU(
    (W_r): Linear(in_features=100, out_features=128, bias=True)
    (U_r): Linear(in_features=128, out_features=128, bias=True)
    (W_z): Linear(in_features=100, out_features=128, bias=True)
    (U_z): Linear(in_features=128, out_features=128, bias=True)
    (W_h): Linear(in_features=100, out_features=128, bias=True)
    (U_h): Linear(in_features=128, out_features=128, bias=True)
  )
  (attention): AttentionLayer()
  (fc): Linear(in_features=256, out_features=15, bias=True)
)

In [27]:
torch.cuda.empty_cache()

In [28]:
def predict_classes(model, X_tensor):
    with torch.no_grad():
        outputs = model(X_tensor)
        _, preds = torch.max(outputs, 1)
    return preds.cpu().numpy()

def evaluate_all_models(models_dict, X_tensor_test, y_test):
    results = {}
    
    for name, model in models_dict.items():
        preds = predict_classes(model, X_tensor_test)
        
        results[name] = {
            'accuracy': accuracy_score(y_test, preds),
            'f1': f1_score(y_test, preds, average='weighted', zero_division=0),
            'precision': precision_score(y_test, preds, average='weighted', zero_division=0),
            'recall': recall_score(y_test, preds, average='weighted', zero_division=0)
        }

    return results

In [29]:
X_tensor_test, _ = prepare_data(X_test, y_test.values)
X_tensor_test = X_tensor_test.to(device)

In [30]:
models_with_attention = {
    'RNN': rnn_model,
    'RNN_Attn': rnn_attention_model,
    'LSTM': lstm_model,
    'LSTM_Attn': lstm_attention_model,
    'GRU': gru_model,
    'GRU_Attn': gru_attention_model,
    'C_GRU': custom_gru_model,
    'C_GRU_Attn': custom_gru_attention_model
}

results_all = evaluate_all_models(models_with_attention, X_tensor_test, y_test)
results_df = pd.DataFrame.from_dict(results_all, orient='index')
results_df.round(3)

,accuracy,f1,precision,recall
RNN,0.751,0.724,0.701,0.751
RNN_Attn,0.838,0.827,0.822,0.838
LSTM,0.811,0.797,0.788,0.811
LSTM_Attn,0.847,0.840,0.839,0.847
GRU,0.829,0.826,0.827,0.829
GRU_Attn,0.864,0.855,0.855,0.864
C_GRU,0.822,0.812,0.814,0.822
C_GRU_Attn,0.840,0.836,0.836,0.840
